# 04 — Topic Modelling with BERTopic
**Objective 4:** Discover major themes in guest reviews using transformer embeddings + clustering.

> **Note:** First run may download the embedding model (`all-MiniLM-L6-v2`).  
> On CPU, start with a sample (`SAMPLE_SIZE = 10000`) then set to `None` for full data.


In [1]:
# %pip install bertopic sentence-transformers umap-learn hdbscan

from pathlib import Path

import numpy as np
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
SAMPLE_SIZE = 10000  # set to None for full dataset

def load_cleaned():
    for name in ("cleaned_reviews.parquet", "cleaned_reviews.pkl"):
        p = DATA_DIR / name
        if p.exists() and p.stat().st_size > 0:
            return pd.read_parquet(p) if name.endswith(".parquet") else pd.read_pickle(p)
    raise FileNotFoundError("Run 02_data_cleaning first")

df = load_cleaned()
if SAMPLE_SIZE and SAMPLE_SIZE < len(df):
    docs_df = df.sample(SAMPLE_SIZE, random_state=RANDOM_SEED).copy()
    print(f"Sampled {SAMPLE_SIZE} of {len(df)} reviews")
else:
    docs_df = df.copy()
    print(f"Using all {len(docs_df)} reviews")

documents = docs_df["review_text_clean"].tolist()


ModuleNotFoundError: No module named 'bertopic'

In [ ]:
# Build BERTopic model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

umap_model = UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0,
    metric="cosine", random_state=RANDOM_SEED,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=50, metric="euclidean",
    cluster_selection_method="eom", prediction_data=True,
)
vectorizer_model = CountVectorizer(
    stop_words="english", ngram_range=(1, 2), min_df=5, max_df=0.85,
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    top_n_words=10,
    calculate_probabilities=True,
    verbose=True,
)
print("BERTopic model constructed.")


In [ ]:
# Fit
topics, probs = topic_model.fit_transform(documents)
print(f"Discovered {len(set(topics)) - (1 if -1 in topics else 0)} topics (excl. outliers)")
print(f"Outlier share: {(np.array(topics) == -1).mean():.1%}")

topic_info = topic_model.get_topic_info()
display(topic_info.head(15))


In [ ]:
# Human-readable labels (top-3 words)
labels = {}
for tid in topic_model.get_topics():
    if tid == -1:
        labels[tid] = "Outlier / Mixed"
    else:
        words = [w for w, _ in topic_model.get_topic(tid)[:3]]
        labels[tid] = " / ".join(words)

docs_df["topic_id"] = topics
docs_df["topic_label"] = [labels.get(t, "Unknown") for t in topics]
if probs is not None:
    docs_df["topic_prob"] = [
        float(probs[i][topics[i]]) if topics[i] != -1 else 0.0
        for i in range(len(topics))
    ]
else:
    docs_df["topic_prob"] = 0.0

docs_df[["review_id", "topic_id", "topic_label", "topic_prob"]].head(10)


In [ ]:
# Inspect a few topics
for tid in topic_info["Topic"].head(8):
    if tid == -1:
        continue
    print(f"\n=== Topic {tid}: {labels[tid]} ===")
    print(topic_model.get_topic(tid)[:8])


In [ ]:
# Save model + document-topic mapping
topic_model.save(str(MODELS_DIR / "bertopic_model"), serialization="safetensors", save_ctfidf=True)
topic_info["label"] = topic_info["Topic"].map(labels)
topic_info.to_csv(MODELS_DIR / "topic_info.csv", index=False)

mapping = docs_df[["review_id", "topic_id", "topic_label", "topic_prob"]]
try:
    mapping.to_parquet(MODELS_DIR / "document_topics.parquet", index=False)
except Exception:
    mapping.to_pickle(MODELS_DIR / "document_topics.pkl")
print("Saved BERTopic model and document-topic mapping.")


In [ ]:
# Optional visualisations
try:
    fig = topic_model.visualize_barchart(top_n_topics=12)
    fig.show()
except Exception as e:
    print("Interactive plot skipped:", e)
